# ETL Pipeline: Big Data Applications (China firm-year panel)
**Extract → Transform → Load (Excel)**

| Step | What happens |
|---|---|
| **Extract** | Read `Big_Data_Applications_chinse.csv` and profile it |
| **Transform** | Validate, clean, drop constant columns, add province / region / sector labels, engineer features |
| **Load** | Write a formatted multi-sheet Excel workbook (`Big_Data_Applications_ETL.xlsx`) + a clean CSV |

Change the paths in the next cell if you run this outside the original folder.

In [15]:
%pip install XlsxWriter

  Using cached xlsxwriter-3.2.9-py3-none-any.whl.metadata (2.7 kB)
Using cached xlsxwriter-3.2.9-py3-none-any.whl (175 kB)
Note: you may need to restart the kernel to use updated packages.


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_CSV  = Path("Big_Data_Applications_chinse.csv")
OUTPUT_XLSX = Path("Big_Data_Applications_ETL.xlsx")
OUTPUT_CSV  = Path("Big_Data_Applications_clean.csv")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 1. EXTRACT

In [17]:
raw = pd.read_csv(INPUT_CSV, encoding="utf-8")
print(f"Rows: {raw.shape[0]:,}   Columns: {raw.shape[1]}")
raw.head()

Rows: 21,203   Columns: 15


,code,year,TD,MD,BDA,province,Myopia,IC,AC,ADC,GI,GP,ind3,TD_std,MD_std
0,8,2015,119,890,4,110000,0,0,0,-1,0,1,C37,0,0
1,8,2016,98,691,4,110000,0,1,0,-1,0,2,C37,0,0
2,8,2017,133,978,5,110000,0,1,0,-1,0,2,C37,0,0
3,8,2018,97,722,5,110000,0,1,0,-1,0,2,C37,0,0
4,8,2019,173,1309,5,110000,0,1,0,-1,0,1,C37,0,0


In [18]:
# Data profile
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_missing": raw.isna().sum(),
    "n_unique": raw.nunique(),
    "min": raw.min(numeric_only=True),
    "max": raw.max(numeric_only=True),
})
print("Duplicate rows:", raw.duplicated().sum())
print("Duplicate (code, year) keys:", raw.duplicated(["code", "year"]).sum())
profile

Duplicate rows: 0
Duplicate (code, year) keys: 0


,dtype,n_missing,n_unique,min,max
AC,int64,0,1,0.0,0.0
ADC,int64,0,3,-2.0,0.0
BDA,int64,0,7,1.0,7.0
GI,int64,0,2,0.0,1.0
GP,int64,0,9,0.0,8.0
IC,int64,0,3,0.0,2.0
MD,int64,0,3251,8.0,14499.0
MD_std,int64,0,8,0.0,7.0
Myopia,int64,0,1,0.0,0.0
TD,int64,0,1007,1.0,1942.0


## 2. TRANSFORM

### 2.1 Data-quality checks

In [19]:
checks = {
    "No missing values":              raw.isna().sum().sum() == 0,
    "No duplicate rows":              raw.duplicated().sum() == 0,
    "(code, year) is a unique key":   not raw.duplicated(["code", "year"]).any(),
    "TD and MD are positive":         bool((raw[["TD", "MD"]] > 0).all().all()),
    "TD <= MD in every row":          bool((raw["TD"] <= raw["MD"]).all()),
    "Years within 2014-2023":         raw["year"].between(2014, 2023).all(),
    "Province codes are 6 digits":    raw["province"].between(100000, 999999).all(),
}
qc = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
qc

,check,passed
0,No missing values,True
1,No duplicate rows,True
2,"(code, year) is a unique key",True
3,TD and MD are positive,True
4,TD <= MD in every row,True
5,Years within 2014-2023,True
6,Province codes are 6 digits,True


### 2.2 Clean

In [20]:
df = raw.copy()

# 1) Standardise column names to snake_case-friendly names (originals kept in the data dictionary)
df = df.rename(columns={"code": "firm_code", "ind3": "industry_code"})

# 2) Zero-variance columns carry no information -> drop them
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns dropped:", const_cols)
df = df.drop(columns=const_cols)

# 3) Types
df["firm_code"]     = df["firm_code"].astype("int32")
df["year"]          = df["year"].astype("int16")
df["province"]      = df["province"].astype("int32")
df["industry_code"] = df["industry_code"].str.strip().str.upper()
for c in ["TD", "MD"]:
    df[c] = df[c].astype("int32")

# 4) Firm codes are stock tickers -> keep a zero-padded 6-char text version for joins/lookups
df["firm_code_str"] = df["firm_code"].astype(str).str.zfill(6)

# 5) Sort into panel order
df = df.sort_values(["firm_code", "year"]).reset_index(drop=True)
df.head()

Constant columns dropped: ['Myopia', 'AC']


,firm_code,year,TD,MD,BDA,province,IC,ADC,GI,GP,industry_code,TD_std,MD_std,firm_code_str
0,8,2015,119,890,4,110000,0,-1,0,1,C37,0,0,000008
1,8,2016,98,691,4,110000,1,-1,0,2,C37,0,0,000008
2,8,2017,133,978,5,110000,1,-1,0,2,C37,0,0,000008
3,8,2018,97,722,5,110000,1,-1,0,2,C37,0,0,000008
4,8,2019,173,1309,5,110000,1,-1,0,1,C37,0,0,000008


### 2.3 Enrich with lookup tables (province names, regions, sector)

In [21]:
# GB/T 2260 province-level codes
PROVINCES = {
    110000: ("Beijing", "East"),       120000: ("Tianjin", "East"),
    130000: ("Hebei", "East"),         140000: ("Shanxi", "Central"),
    150000: ("Inner Mongolia", "West"),210000: ("Liaoning", "Northeast"),
    220000: ("Jilin", "Northeast"),    230000: ("Heilongjiang", "Northeast"),
    310000: ("Shanghai", "East"),      320000: ("Jiangsu", "East"),
    330000: ("Zhejiang", "East"),      340000: ("Anhui", "Central"),
    350000: ("Fujian", "East"),        360000: ("Jiangxi", "Central"),
    370000: ("Shandong", "East"),      410000: ("Henan", "Central"),
    420000: ("Hubei", "Central"),      430000: ("Hunan", "Central"),
    440000: ("Guangdong", "East"),     450000: ("Guangxi", "West"),
    460000: ("Hainan", "East"),        500000: ("Chongqing", "West"),
    510000: ("Sichuan", "West"),       520000: ("Guizhou", "West"),
    530000: ("Yunnan", "West"),        540000: ("Tibet", "West"),
    610000: ("Shaanxi", "West"),       620000: ("Gansu", "West"),
    630000: ("Qinghai", "West"),       640000: ("Ningxia", "West"),
    650000: ("Xinjiang", "West"),
}
prov_lookup = (pd.DataFrame.from_dict(PROVINCES, orient="index", columns=["province_name", "region"])
                 .rename_axis("province").reset_index())

df = df.merge(prov_lookup, on="province", how="left", validate="m:1")
print("Provinces without a name:", df["province_name"].isna().sum())

# Industry: first letter = CSRC sector (e.g. C = Manufacturing), number = sub-industry
df["sector"] = df["industry_code"].str[0]
df[["firm_code", "year", "province", "province_name", "region", "industry_code", "sector"]].head()

Provinces without a name: 0


,firm_code,year,province,province_name,region,industry_code,sector
0,8,2015,110000,Beijing,East,C37,C
1,8,2016,110000,Beijing,East,C37,C
2,8,2017,110000,Beijing,East,C37,C
3,8,2018,110000,Beijing,East,C37,C
4,8,2019,110000,Beijing,East,C37,C


### 2.4 Feature engineering

In [22]:
g = df.groupby("firm_code", sort=False)

df["log_TD"]   = np.log1p(df["TD"])
df["log_MD"]   = np.log1p(df["MD"])
df["TD_share"] = (df["TD"] / df["MD"]).round(4)          # TD as a share of MD

# Year-over-year growth within each firm
df["TD_growth"] = g["TD"].pct_change().round(4)
df["MD_growth"] = g["MD"].pct_change().round(4)

# Change in BDA vs. previous year
df["BDA_change"] = g["BDA"].diff()

# Panel descriptors
df["n_years_observed"] = g["year"].transform("count")
df["first_year"]       = g["year"].transform("min")
df["last_year"]        = g["year"].transform("max")

# Replace inf produced by growth rates (none expected, but be safe)
df = df.replace([np.inf, -np.inf], np.nan)
df.head()

,firm_code,year,TD,MD,BDA,province,IC,ADC,GI,GP,industry_code,TD_std,MD_std,firm_code_str,province_name,region,sector,log_TD,log_MD,TD_share,TD_growth,MD_growth,BDA_change,n_years_observed,first_year,last_year
0,8,2015,119,890,4,110000,0,-1,0,1,C37,0,0,000008,Beijing,East,C,4.787492,6.792344,0.1337,NaN,NaN,NaN,9,2015,2023
1,8,2016,98,691,4,110000,1,-1,0,2,C37,0,0,000008,Beijing,East,C,4.595120,6.539586,0.1418,-0.1765,-0.2236,0.0,9,2015,2023
2,8,2017,133,978,5,110000,1,-1,0,2,C37,0,0,000008,Beijing,East,C,4.897840,6.886532,0.1360,0.3571,0.4153,1.0,9,2015,2023
3,8,2018,97,722,5,110000,1,-1,0,2,C37,0,0,000008,Beijing,East,C,4.584967,6.583409,0.1343,-0.2707,-0.2618,0.0,9,2015,2023
4,8,2019,173,1309,5,110000,1,-1,0,1,C37,0,0,000008,Beijing,East,C,5.159055,7.177782,0.1322,0.7835,0.8130,0.0,9,2015,2023


### 2.5 Aggregate / summary tables

In [23]:
by_year = (df.groupby("year")
             .agg(firms=("firm_code", "nunique"),
                  TD_total=("TD", "sum"), MD_total=("MD", "sum"),
                  TD_mean=("TD", "mean"), MD_mean=("MD", "mean"),
                  BDA_mean=("BDA", "mean"), GI_share=("GI", "mean"))
             .round(3).reset_index())

by_province = (df.groupby(["province", "province_name", "region"])
                 .agg(firms=("firm_code", "nunique"), obs=("year", "size"),
                      TD_mean=("TD", "mean"), MD_mean=("MD", "mean"),
                      BDA_mean=("BDA", "mean"), GI_share=("GI", "mean"))
                 .round(3).reset_index().sort_values("firms", ascending=False))

by_industry = (df.groupby("industry_code")
                 .agg(firms=("firm_code", "nunique"), obs=("year", "size"),
                      TD_mean=("TD", "mean"), MD_mean=("MD", "mean"),
                      BDA_mean=("BDA", "mean"))
                 .round(3).reset_index().sort_values("firms", ascending=False))

by_region_year = (df.pivot_table(index="region", columns="year", values="BDA", aggfunc="mean")
                    .round(3).reset_index())

prov_year_TD = (df.pivot_table(index="province_name", columns="year", values="TD", aggfunc="sum")
                  .fillna(0).astype(int).reset_index())

by_year

,year,firms,TD_total,MD_total,TD_mean,MD_mean,BDA_mean,GI_share
0,2014,1253,123737,898189,98.753,716.831,3.955,0.054
1,2015,1346,139746,1014797,103.823,753.935,3.749,0.057
2,2016,1472,163104,1184122,110.804,804.431,3.923,0.058
3,2017,1782,196492,1427308,110.265,800.958,4.137,0.066
4,2018,1921,239444,1741337,124.645,906.474,4.184,0.066
5,2019,2035,275551,2004350,135.406,984.939,4.271,0.083
6,2020,2318,321225,2334805,138.579,1007.250,4.364,0.077
7,2021,2724,355854,2590574,130.637,951.018,4.429,0.047
8,2022,3024,377174,2748432,124.727,908.873,4.435,0.045
9,2023,3328,381194,2793239,114.541,839.315,4.554,0.048


In [24]:
by_province.head(10)

,province,province_name,region,firms,obs,TD_mean,MD_mean,BDA_mean,GI_share
18,440000,Guangdong,East,613,3596,159.738,1157.627,4.428,0.0
10,330000,Zhejiang,East,558,3231,83.162,609.305,4.222,0.0
9,320000,Jiangsu,East,530,2910,81.834,596.526,4.248,0.0
8,310000,Shanghai,East,244,1397,126.588,921.586,4.273,0.0
14,370000,Shandong,East,231,1448,140.821,1033.669,4.194,0.0
0,110000,Beijing,East,176,1136,233.463,1690.788,4.452,0.0
11,340000,Anhui,Central,128,742,151.221,1097.889,4.249,0.0
22,510000,Sichuan,West,117,708,110.335,794.586,4.284,0.0
12,350000,Fujian,East,102,635,105.586,756.712,4.299,0.0
16,420000,Hubei,Central,95,576,148.828,1063.995,4.297,0.0


### 2.6 Data dictionary

In [25]:
dictionary = pd.DataFrame([
    ("firm_code",        "Original `code`: firm identifier (numeric stock code)"),
    ("firm_code_str",    "Firm code as zero-padded 6-character text"),
    ("year",             "Observation year (2014-2023)"),
    ("TD, MD",           "Original count variables (TD <= MD in every row); names kept as in source"),
    ("BDA",              "Big Data Applications score (integer 1-7)"),
    ("province",         "Original GB/T 2260 province code"),
    ("province_name",    "Added: province name from lookup"),
    ("region",           "Added: East / Central / West / Northeast grouping"),
    ("IC, ADC, GI, GP",  "Original indicator / score columns, kept unchanged"),
    ("industry_code",    "Original `ind3`: industry code"),
    ("sector",           "Added: first letter of industry_code"),
    ("TD_std, MD_std",   "Original standardised-count columns, kept unchanged"),
    ("log_TD, log_MD",   "Added: log(1 + x)"),
    ("TD_share",         "Added: TD / MD"),
    ("TD_growth, MD_growth", "Added: year-over-year % change within firm"),
    ("BDA_change",       "Added: change in BDA vs. previous year, within firm"),
    ("n_years_observed, first_year, last_year", "Added: panel coverage per firm"),
    ("Dropped",          ", ".join(const_cols) + " (constant, zero information)"),
], columns=["column", "description"])
dictionary

,column,description
0,firm_code,Original `code`: firm identifier (numeric stoc...
1,firm_code_str,Firm code as zero-padded 6-character text
2,year,Observation year (2014-2023)
3,"TD, MD",Original count variables (TD <= MD in every ro...
4,BDA,Big Data Applications score (integer 1-7)
5,province,Original GB/T 2260 province code
6,province_name,Added: province name from lookup
7,region,Added: East / Central / West / Northeast grouping
8,"IC, ADC, GI, GP","Original indicator / score columns, kept uncha..."
9,industry_code,Original `ind3`: industry code


## 3. LOAD → Excel

In [26]:
final_cols = ["firm_code", "firm_code_str", "year", "province", "province_name", "region",
              "industry_code", "sector", "TD", "MD", "BDA", "IC", "ADC", "GI", "GP",
              "TD_std", "MD_std", "log_TD", "log_MD", "TD_share",
              "TD_growth", "MD_growth", "BDA_change",
              "n_years_observed", "first_year", "last_year"]
clean = df[final_cols]

sheets = {
    "Clean_Data":       clean,
    "By_Year":          by_year,
    "By_Province":      by_province,
    "By_Industry":      by_industry,
    "BDA_Region_Year":  by_region_year,
    "TD_Province_Year": prov_year_TD,
    "Quality_Checks":   qc,
    "Data_Dictionary":  dictionary,
}

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as xw:
    wb = xw.book
    hdr = wb.add_format({"bold": True, "bg_color": "#1F4E78", "font_color": "white",
                         "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
    for name, d in sheets.items():
        d.to_excel(xw, sheet_name=name, index=False)
        ws = xw.sheets[name]
        for i, col in enumerate(d.columns):
            ws.write(0, i, col, hdr)
            width = min(max(len(str(col)), d[col].astype(str).str.len().max()) + 2, 60)
            ws.set_column(i, i, width)
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(d), len(d.columns) - 1)

    # Chart: mean BDA by year
    ws = xw.sheets["By_Year"]
    ch = wb.add_chart({"type": "line"})
    n = len(by_year)
    col_bda = by_year.columns.get_loc("BDA_mean")
    ch.add_series({"name": "Mean BDA",
                   "categories": ["By_Year", 1, 0, n, 0],
                   "values":     ["By_Year", 1, col_bda, n, col_bda],
                   "marker": {"type": "circle"}})
    ch.set_title({"name": "Average BDA by year"})
    ch.set_legend({"none": True})
    ws.insert_chart("K2", ch)

clean.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_XLSX, "and", OUTPUT_CSV)

Saved: Big_Data_Applications_ETL.xlsx and Big_Data_Applications_clean.csv


## 4. Verify the load

In [27]:
check = pd.read_excel(OUTPUT_XLSX, sheet_name=None)
pd.DataFrame({"sheet": list(check), "rows": [len(v) for v in check.values()],
              "columns": [v.shape[1] for v in check.values()]})

,sheet,rows,columns
0,Clean_Data,21203,26
1,By_Year,10,8
2,By_Province,31,9
3,By_Industry,32,6
4,BDA_Region_Year,4,11
5,TD_Province_Year,31,11
6,Quality_Checks,7,2
7,Data_Dictionary,18,2


In [28]:
assert len(check["Clean_Data"]) == len(raw), "Row count changed during ETL"
assert not check["Clean_Data"].duplicated(["firm_code", "year"]).any()
print(f"OK: {len(raw):,} rows in -> {len(check['Clean_Data']):,} rows out")

OK: 21,203 rows in -> 21,203 rows out
